In [2]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

session = requests.Session()
resp = session.post(
    "https://www.space-track.org/ajaxauth/login",
    data={"identity": os.getenv("ST_USER"), "password": os.getenv("ST_PASS")},
    timeout=15  # fail fast instead of hanging forever
)
print(f"Login status: {resp.status_code}")

# Small test query with an explicit timeout
url = "https://www.space-track.org/basicspacedata/query/class/gp/EPOCH/%3Enow-30/OBJECT_TYPE/PAYLOAD/orderby/NORAD_CAT_ID/format/tle/limit/5"
resp2 = session.get(url, timeout=20)
print(f"Query status: {resp2.status_code}")
print(resp2.text[:200])

Login status: 200
Query status: 200
1 00005U 58002B   26191.18028143  .00000196  00000-0  22672-3 0  9994
2 00005  34.2532 243.3685 1836938 242.1885  98.0797 10.86014642445785
1 00011U 59001A   26190.87062961  .00000980  00000-0  5008


In [3]:
url = (
    "https://www.space-track.org/basicspacedata/query/class/gp/"
    "EPOCH/%3Enow-30/OBJECT_TYPE/PAYLOAD/MEAN_MOTION/%3E11.25/"
    "orderby/NORAD_CAT_ID/format/tle/limit/5000"
)

import time
start = time.time()
resp = session.get(url, timeout=60)
elapsed = time.time() - start

print(f"Status: {resp.status_code}")
print(f"Completed in {elapsed:.1f} seconds")

lines = resp.text.strip().splitlines()
print(f"Total lines: {len(lines)}")
print(f"Total objects: {len(lines) // 2}")

with open("data/tle_raw_5k.txt", "w") as f:
    f.write(resp.text)
print("Saved to data/tle_raw_5k.txt")

Status: 200
Completed in 0.6 seconds
Total lines: 10000
Total objects: 5000
Saved to data/tle_raw_5k.txt


In [4]:
from sgp4.api import Satrec, jday
from datetime import datetime, timedelta
import numpy as np
import time

with open("data/tle_raw_5k.txt", "r") as f:
    lines_5k = f.read().strip().splitlines()

satellites_5k = []
for i in range(0, len(lines_5k), 2):
    sat = Satrec.twoline2rv(lines_5k[i], lines_5k[i + 1])
    satellites_5k.append(sat)

print(f"Parsed {len(satellites_5k)} satellites")

# Same 24-hour window, 5-minute steps as before
start_time = datetime.utcnow()
time_step_minutes = 5
num_steps = 288
time_points_5k = [start_time + timedelta(minutes=i * time_step_minutes) for i in range(num_steps)]

print(f"\nPropagating {len(satellites_5k)} satellites across {num_steps} time steps...")
start = time.time()

all_positions_5k = np.zeros((len(satellites_5k), num_steps, 3))
all_errors_5k = np.zeros((len(satellites_5k), num_steps), dtype=int)

for sat_idx, sat in enumerate(satellites_5k):
    for t_idx, t in enumerate(time_points_5k):
        jd, fr = jday(t.year, t.month, t.day, t.hour, t.minute, t.second)
        error, position, velocity = sat.sgp4(jd, fr)
        all_positions_5k[sat_idx, t_idx] = position
        all_errors_5k[sat_idx, t_idx] = error

elapsed = time.time() - start
print(f"Propagation completed in {elapsed:.1f} seconds")
print(f"Positions array shape: {all_positions_5k.shape}")
print(f"Errors: {np.sum(all_errors_5k != 0)} out of {all_errors_5k.size}")

Parsed 5000 satellites

Propagating 5000 satellites across 288 time steps...
Propagation completed in 1.3 seconds
Positions array shape: (5000, 288, 3)
Errors: 15829 out of 1440000


In [5]:
valid_mask_5k = ~np.any(all_errors_5k != 0, axis=1)

clean_satellites_5k = [sat for i, sat in enumerate(satellites_5k) if valid_mask_5k[i]]
clean_positions_5k = all_positions_5k[valid_mask_5k]
clean_indices_5k = np.where(valid_mask_5k)[0]

norad_ids_5k = [sat.satnum for sat in clean_satellites_5k]

print(f"Kept {len(clean_satellites_5k)} clean satellites out of {len(satellites_5k)}")
print(f"Clean positions shape: {clean_positions_5k.shape}")

Kept 4944 clean satellites out of 5000
Clean positions shape: (4944, 288, 3)


In [6]:
from scipy.spatial import cKDTree
import time

SCREENING_DISTANCE_KM = 25.0

close_approaches_5k = []

start = time.time()
for t_idx in range(clean_positions_5k.shape[1]):
    positions_at_t = clean_positions_5k[:, t_idx, :]
    tree = cKDTree(positions_at_t)
    pairs = tree.query_pairs(r=SCREENING_DISTANCE_KM)
    
    for i, j in pairs:
        dist = np.linalg.norm(positions_at_t[i] - positions_at_t[j])
        close_approaches_5k.append((i, j, t_idx, dist))

elapsed = time.time() - start
print(f"Screening completed in {elapsed:.1f} seconds")
print(f"Found {len(close_approaches_5k)} close-approach events within {SCREENING_DISTANCE_KM} km")

import pandas as pd
df_5k = pd.DataFrame(close_approaches_5k, columns=["sat_i", "sat_j", "time_idx", "distance_km"])
closest_per_pair_5k = df_5k.loc[df_5k.groupby(["sat_i", "sat_j"])["distance_km"].idxmin()]
closest_per_pair_5k = closest_per_pair_5k.sort_values("distance_km").reset_index(drop=True)

print(f"\nUnique satellite pairs with a close approach: {len(closest_per_pair_5k)}")
print(f"\nDistribution of closest distances:")
print(closest_per_pair_5k["distance_km"].describe())

Screening completed in 0.4 seconds
Found 6898 close-approach events within 25.0 km

Unique satellite pairs with a close approach: 829

Distribution of closest distances:
count    829.000000
mean      17.236838
std        5.779463
min        0.000000
25%       13.637846
50%       18.382082
75%       21.843246
max       24.993397
Name: distance_km, dtype: float64


In [7]:
# Step 6a: Extract orbital elements for every clean satellite (no propagation needed - these come directly from the TLE)
orbital_features = []
for sat in clean_satellites_5k:
    orbital_features.append({
        "norad_id": sat.satnum,
        "inclination": sat.inclo,      # radians
        "raan": sat.nodeo,              # radians
        "eccentricity": sat.ecco,
        "arg_perigee": sat.argpo,       # radians
        "mean_motion": sat.no_kozai,    # radians/minute
        "mean_anomaly": sat.mo          # radians
    })

orbit_df = pd.DataFrame(orbital_features)
print(f"Extracted orbital elements for {len(orbit_df)} satellites")
print(orbit_df.head())

Extracted orbital elements for 4944 satellites
   norad_id  inclination      raan  eccentricity  arg_perigee  mean_motion  \
0        11     0.573716  6.270673      0.144521     3.247026     0.051949   
1        20     0.581895  3.222487      0.163746     3.074153     0.050701   
2        22     0.876928  6.037040      0.007734     3.809879     0.066730   
3        29     0.844359  4.857141      0.002203     0.063298     0.064525   
4        45     1.164040  5.255862      0.024408     2.518257     0.062657   

   mean_anomaly  
0      3.003394  
1      3.235481  
2      2.465374  
3      6.221755  
4      3.795838  
